In [2]:
# RAG Simple - Processing the Paper "Attention Is All You Need"
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
EMB_MODEL = os.getenv("EMB_MODEL", "nomic-embed-text:latest")
embeddings = OllamaEmbeddings(model=EMB_MODEL)

/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
/var/folders/t3/97hgmq6x6mg3dybs2fbsfcqr0000gn/T/ipykernel_23826/1028193504.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
try:
    # Intentar OpenAI primero
    llm = ChatOpenAI(
        model=OPENAI_MODEL,
        temperature=0,
        api_key=os.getenv("OPENAI_API_KEY"),
    )
    _ = llm.invoke("ping")
    provider = "OpenAI"
except Exception:
    # Si OpenAI falla, usar Ollama
    llm = ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0,
    )
    _ = llm.invoke("ping")
    provider = "Ollama"

print(f"Proveedor activo: {provider}")
response = llm.invoke("What is the Transformer architecture?")
response.pretty_print()

Proveedor activo: OpenAI
================================== Ai Message ==================================

The Transformer architecture is a type of neural network architecture that was introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017. It has become a foundational model for various natural language processing (NLP) tasks and has also been adapted for other domains such as computer vision and audio processing. The key features of the Transformer architecture include:

### 1. **Self-Attention Mechanism:**
   - The core innovation of the Transformer is the self-attention mechanism, which allows the model to weigh the importance of different words in a sentence relative to each other. This enables the model to capture contextual relationships without relying on sequential processing, as seen in recurrent neural networks (RNNs).

### 2. **Positional Encoding:**
   - Since Transformers do not have a built-in notion of sequence order (unlike RNNs), they use positi

# **Build RAG** 

In [4]:
# Setup RAG
cache = "../faiss_cache/transformer_paper"

# Cargar y procesar PDF
docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(
    PyPDFLoader("../pdfs/Paper.pdf").load()
)

# Vector store con cache (embeddings locales con Ollama)
if os.path.exists(cache + ".faiss"):
    vectorstore = FAISS.load_local(
        cache,
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    vectorstore = FAISS.from_documents(docs, embeddings)
    vectorstore.save_local(cache)


In [5]:
PROMPT_CLASSIC = """ 
                    Basándote en el paper 'Attention Is All You Need':
                    
                    ontexto: {context}
                    
                    Pregunta: {question}
                    
                    Respuesta:
""".format(context=context, question=question)

NameError: name 'context' is not defined

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(
    "Basándote en el paper 'Attention Is All You Need':\n\nContexto: {context}\n\nPregunta: {question}\n\nRespuesta:"
)

rag = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print("✅ RAG listo")

✅ RAG listo


In [7]:
type(prompt)

langchain_core.prompts.chat.ChatPromptTemplate

## 🔍 **Make questions**

In [ ]:
# Preguntar directamente
rag.invoke("What is the Transformer architecture?")


'Según el paper "Attention Is All You Need", la arquitectura del Transformer es un modelo que se basa en una atención mecánica para dibujar dependencias globales entre la entrada y la salida, en lugar de utilizar recurrencia. Este modelo permite una mayor paralelización y puede alcanzar un nuevo estado de la arte en la calidad de traducción después de ser entrenado durante como máximo 12 horas con 8 unidades de GPU P100.'

In [ ]:
# O con función helper
def ask(question, show_sources=False):
    result = rag.invoke(question)
    print(f"💡 {result}\n")
    if show_sources:
        docs = retriever.invoke(question)
        print(f"📚 Fuentes ({len(docs)} docs):")
        for i, doc in enumerate(docs, 1):
            print(f"  {i}. Pág {doc.metadata.get('page', '?')}")
    return result

ask("What is multi-head attention?")


💡 Según el texto proporcionado, la respuesta es:

Multi-Head Attention es una forma de atención que permite al modelo jointly atender a información de diferentes subespacios representativos en diferentes posiciones. Esto se logra mediante la concatenación de varias "cabezas" de atención (denotadas como headi) y luego aplicando un proceso de proyección (representado por las matrices WQ, WK, WV y WO).

En otras palabras, Multi-Head Attention es una forma de atención que permite al modelo considerar múltiples subespacios representativos simultáneamente, en lugar de solo considerar uno a la vez. Esto se logra mediante la concatenación de varias "cabezas" de atención, cada una de las cuales aplica un proceso de atención individual (representado por la función Attention(QWQi, KWKi, Vi)).



'Según el texto proporcionado, la respuesta es:\n\nMulti-Head Attention es una forma de atención que permite al modelo jointly atender a información de diferentes subespacios representativos en diferentes posiciones. Esto se logra mediante la concatenación de varias "cabezas" de atención (denotadas como headi) y luego aplicando un proceso de proyección (representado por las matrices WQ, WK, WV y WO).\n\nEn otras palabras, Multi-Head Attention es una forma de atención que permite al modelo considerar múltiples subespacios representativos simultáneamente, en lugar de solo considerar uno a la vez. Esto se logra mediante la concatenación de varias "cabezas" de atención, cada una de las cuales aplica un proceso de atención individual (representado por la función Attention(QWQi, KWKi, Vi)).'